In [4]:
import numpy as np

def truss2D(nodes, elements, E, A, forces, fixed_dofs):
    """
    2D Truss Finite Element Solver

    Parameters
    ----------
    nodes : array (n_nodes x 2)
        Node coordinates [[x1,y1],[x2,y2],...]
    
    elements : list of tuples
        Connectivity list [(i,j), ...] (zero-based indexing)
    
    E : float or list
        Young's modulus (scalar or per element)
    
    A : float or list
        Cross-sectional area (scalar or per element)
    
    forces : array (2*n_nodes)
        Global force vector [Fx0,Fy0,Fx1,Fy1,...]
    
    fixed_dofs : list
        Indices of constrained DOFs

    Returns
    -------
    U : array
        Global displacement vector
    K : array
        Global stiffness matrix
    """

    # -----------------------------------------------------------
    # 1. Determine system size
    # -----------------------------------------------------------
    n_nodes = nodes.shape[0]      # number of nodes
    n_dofs = 2 * n_nodes          # 2 DOFs per node (ux, uy)

    # Initialize global stiffness matrix
    K = np.zeros((n_dofs, n_dofs))

    # -----------------------------------------------------------
    # 2. Allow material properties to be scalar or per-element
    # -----------------------------------------------------------
    if np.isscalar(E):
        E = np.full(len(elements), E)

    if np.isscalar(A):
        A = np.full(len(elements), A)

    # -----------------------------------------------------------
    # 3. Loop through each element and assemble stiffness
    # -----------------------------------------------------------
    for e, (i, j) in enumerate(elements):

        # Node coordinates
        xi, yi = nodes[i]
        xj, yj = nodes[j]

        # Element length
        L = np.sqrt((xj - xi)**2 + (yj - yi)**2)

        # Direction cosines
        c = (xj - xi) / L
        s = (yj - yi) / L

        # -------------------------------------------------------
        # Element stiffness matrix in global coordinates
        # k = (EA/L) * transformation matrix
        # -------------------------------------------------------
        k_local = (E[e] * A[e] / L) * np.array([
            [ c*c,  c*s, -c*c, -c*s],
            [ c*s,  s*s, -c*s, -s*s],
            [-c*c, -c*s,  c*c,  c*s],
            [-c*s, -s*s,  c*s,  s*s]
        ])

        # Map local DOFs to global DOFs
        dof_map = [
            2*i, 2*i+1,   # Node i (ux, uy)
            2*j, 2*j+1    # Node j (ux, uy)
        ]

        # -------------------------------------------------------
        # Assemble into global stiffness matrix
        # -------------------------------------------------------
        for a in range(4):
            for b in range(4):
                K[dof_map[a], dof_map[b]] += k_local[a, b]

    # -----------------------------------------------------------
    # 4. Apply boundary conditions
    # -----------------------------------------------------------

    # Determine free DOFs
    free_dofs = np.setdiff1d(np.arange(n_dofs), fixed_dofs)

    # Reduce stiffness matrix and force vector
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    F_f = forces[free_dofs]

    # -----------------------------------------------------------
    # 5. Solve reduced system: K_ff * U_f = F_f
    # -----------------------------------------------------------
    U = np.zeros(n_dofs)
    U[free_dofs] = np.linalg.solve(K_ff, F_f)

    # -----------------------------------------------------------
    # 6. Return displacement vector and full stiffness matrix
    # -----------------------------------------------------------
    return U, K

In [7]:
import numpy as np

# --- Nodes ---
nodes = np.array([
    [0.0, 0.0],   # Node 0 (fixed)
    [1.0, 0.0],   # Node 1 (loaded)
    [0.0, 1.0]    # Node 2 (fixed)
])

# --- Elements ---
elements = [(0,1),(1,2),(2,0)]

# --- Material ---
E = 200e9
A = .10

# --- Forces ---
forces = np.zeros(2*len(nodes))

# Node 1 forces
forces[2] = 1    # Fx at node 1
forces[3] = 1    # Fy at node 1

# --- Boundary conditions ---
# Node 0 fixed (DOF 0,1)
# Node 2 fixed (DOF 4,5)
fixed_dofs = [0,1,4,5]

# --- Solve ---
U, K = truss2D(nodes, elements, E, A, forces, fixed_dofs)

print("Displacements:")
print(U.reshape(-1,2))

Displacements:
[[0.00000000e+00 0.00000000e+00]
 [1.00000000e-10 2.41421356e-10]
 [0.00000000e+00 0.00000000e+00]]
